In [272]:
import numpy as np
import pandas as pd
import altair as alt
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
tqdm.pandas()
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [385]:
from clickhouse_driver import Client as Clickhouse
from uuid import uuid4
from pathlib import Path

def click_query_fsn(q, params=None):
    click = Clickhouse.from_url('clickhouse://backend-fsn.ooni.org:9000/default')
    return click.query_dataframe(q, params=params)

def click_query(q, **kw):
    click = Clickhouse("localhost")
    query_id = f"oonidata-{uuid4()}"
    print(f"Starting query_id: {query_id}")
    return click.query_dataframe(q, params=kw, query_id=query_id)

In [386]:
columns = click_query_fsn("""DESCRIBE TABLE fastpath""")

In [387]:
columns

,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,measurement_uid,String,,,,,
1,report_id,String,,,,,
2,input,String,,,,,
3,probe_cc,LowCardinality(String),,,,,
4,probe_asn,Int32,,,,,
5,test_name,LowCardinality(String),,,,,
6,test_start_time,DateTime,,,,,
7,measurement_start_time,DateTime,,,,,
8,filename,String,,,,,
9,scores,String,,,,,


In [388]:
# see: https://learn.microsoft.com/en-us/windows/win32/winsock/windows-sockets-error-codes-2
unknown_failure_map = {
    ': server misbehaving': 'dns_server_misbehaving',
    ': read: connection refused': 'connection_refused',
    ': connect: network is unreachable': 'network_unreachable',
    'tls: first record does not look like a TLS handshake': 'tls_bad_first_record',
    'remote error: tls: handshake failure': 'tls_handshake_failure',
    'remote error: tls: illegal parameter': 'tls_illegal_parameter',
    'connectex: No connection could be made because the target machine actively refused it': 'connection_refused',
    'read: connection refused': 'connection_refused',
    'remote error: tls: access denied': 'tls_access_denied',
    'remote error: tls: internal error': 'tls_internal_error',
    'HTTP/1.x transport connection broken: malformed HTTP version': 'http_malformed_response',
    'net/http: timeout awaiting response headers': 'http_timeout',
    'read: operation timed out': 'timed_out',
    'connect: operation timed out': 'timed_out',
    ': No address associated with hostname': 'dns_nxdomain_error',
    ': connect: bad file descriptor': 'bad_file_descriptor',
    'stream error: stream ID': 'http_stream_error',
    
    # This looks more like a golang-bug: https://github.com/golang/go/issues/31259
    'readLoopPeekFailLocked: <nil>': 'http_golang_bug',

    ': connect: no route to host': 'host_unreachable',
    ': connect: cannot assign requested address': 'address_not_available',
    'getaddrinfow: The requested name is valid, but no data of the requested type was found.': 'dns_no_answer',
    'wsarecv: Se ha forzado la interrupción de una conexión existente por el host remoto.': 'connection_reset',
    'wsarecv: An existing connection was forcibly closed by the remote host.': 'connection_reset',
    'wsarecv: Connessione in corso interrotta forzatamente dall\'host remoto.': 'connection_reset',
    'wsarecv: Uma ligação existente foi forçada a fechar pelo anfitrião remoto': 'connection_reset',
    
    'getaddrinfow: Ceci est habituellement une erreur temporaire qui se produit durant la résolution du nom d’hôte et qui signifie que le serveur local n’a pas reçu de réponse d’un serveur faisant autorité': 'dns_temporary_failure',
    'getaddrinfow: Dies ist normalerweise ein zeitweiliger Fehler bei der Auflösung von Hostnamen. Grund ist, dass der lokale Server keine Rückmeldung vom autorisierenden Server erhalten hat.': 'dns_temporary_failure',
    'getaddrinfow: Este é geralmente um erro temporário durante a resolução de nomes de anfitrião e significa que o servidor local não recebeu uma resposta de um servidor autoritário': 'dns_temporary_failure',
    'getaddrinfow: Éste es normalmente un error temporal durante la resolución de nombres de host y significa que el servidor local no recibió una respuesta de un servidor autoritativo': 'dns_temporary_failure',
}
def map_unknown_failure(failure_str):
    if not failure_str.startswith("unknown_failure"):
        return failure_str
    for substring, clean_failure in unknown_failure_map.items():
        if substring in failure_str:
            return clean_failure
    return "unknown_failure"

def simplify_failure(failure_str):
    if failure_str in ['timed_out', 'generic_timeout_error', 'deferred_timeout_error']:
        return 'timeout'
    
    if failure_str in ['android_dns_cache_no_data', 'dns_nxdomain_error']:
        return 'nxdomain'
    
    if failure_str in ['connection_refused', 'connection_refused_error']:
        return 'connection_refused'
        
    return failure_str

ipv6_failures = ['address_not_available', 'address_family_not_supported', 'network_unreachable', 'host_unreachable']
def compute_analysis(row):
    failure_str = map_unknown_failure(row['failure_str_raw'])

    #if not pd.isnull(row['dns_answer']) and row['dns_answer'] in known_block_ips:
    #    return 'dns.confirmed'
    
    
    if row['tls_is_certificate_valid'] == True:
        return 'ok'

    if row['failure_class'] == 'ok':
        return 'ok'

#     if row['tls_failure'] == 'generic_timeout_error' and row['tls_handshake_last_operation'] == 'write_1':
#         return 'tls.timeout_after_client_hello'
    
    #if row['dns_consistency'] == 'inconsistent':
    #    return 'dns.inconsistent'
    
    if row['ip_as_org_name'] == 'Bogon':
        return 'dns.bogon'
    
    #if row['dns_blocking_scope'] not in ('u', 'n'):
    #    return f"dns.{row['dns_blocking_scope']}"
    
    if row['tls_is_certificate_valid'] == False:
        return 'tls.bad_cert'
    
    simple_failure = simplify_failure(failure_str)
    if simple_failure in ipv6_failures and row['ip'] and ':' in row['ip']:
        return 'ipv6_error'

    if simple_failure.startswith("ssl_"):
        simple_failure = 'bad_cert'
    
    prefix = row['failure_class']
    if prefix == 'https' and row['input'] and row['input'].startswith('https') == False:
        prefix = 'http'
    if prefix == 'https' and simple_failure.startswith('dns_') or simple_failure == 'nxdomain':
        prefix = 'dns'

    return f'{prefix}.{simple_failure}'


In [561]:
ANALYSIS_COUNTRY_CODES = [
    'IR'
]

In [ ]:
# Persian Domains
ANALYSIS_DOMAINS = [
  "iranwire.com",
  "zaagaah.com",
  "www.aasoo.org",
  "cahiersdufeminisme.com",
  "iraws.ir",
  "feminists4jina.net",
  "harasswatch.com",
  "zananemrooz.com",
  "anfpersian.com",
  "jwica.ut.ac.ir",
  "wncri.org",
  "hamamoun.org",
  "feministschool.com",
  "iwontario.com",
  "www.we-change.org",
  "femena.net",
  "avishanx.com"
]

In [562]:
def add_ooni_logo(chart, left_offset, top_offset):
    ooni_logo = alt.Chart(
        {"values": [{"url": "https://raw.githubusercontent.com/ooni/design-system/refs/heads/master/svgs/logos/OONI-HorizontalMonochrome.svg"}]}
    ).mark_image(opacity=0.5).encode(
        x=alt.value(left_offset), x2=alt.value(left_offset+80),  # pixels from left
        y=alt.value(top_offset), y2=alt.value(top_offset+40),    # pixels from top
        url="url:N"
    )

    return alt.vconcat(chart, ooni_logo).configure_concat(
            spacing=-30
        ).configure_view(
            strokeOpacity=0
        )

In [563]:
df_fp_dns = pd.read_csv('https://raw.githubusercontent.com/ooni/blocking-fingerprints/main/fingerprints_dns.csv')
known_block_ips = list(df_fp_dns['pattern'])

In [703]:
START_DAY = '2024-02-01'
END_DAY = '2025-10-01'

In [ ]:
%%time
df = click_query("""
WITH multiIf(
    dns_failure IS NOT NULL, tuple('dns', dns_failure),
    tcp_failure IS NOT NULL, tuple('tcp', tcp_failure),
    tls_failure IS NOT NULL, tuple('tls', tls_failure),
    http_failure IS NOT NULL, tuple('https', http_failure),
    tuple('ok', '')
) as failure
SELECT 
report_id,
input,
measurement_uid,
probe_cc,
probe_asn,
probe_as_org_name,
probe_as_cc,
network_type,
measurement_start_time,
hostname,
ip,
port,
ip_asn,
ip_as_org_name,
resolver_ip,
resolver_cc,
resolver_asn,
resolver_as_org_name,
resolver_as_cc,
dns_engine,
dns_failure,
dns_answer,
tcp_success,
tcp_failure,
tls_handshake_time,
tls_handshake_read_count,
tls_handshake_write_count,
tls_handshake_read_bytes,
tls_handshake_write_bytes,
tls_handshake_last_operation,
tls_cipher_suite IS NOT NULL as tls_success,
tls_is_certificate_valid,
tls_end_entity_certificate_subject,
tls_end_entity_certificate_subject_common_name,
tls_end_entity_certificate_issuer,
tls_end_entity_certificate_issuer_common_name,
tls_end_entity_certificate_san_list,
tls_end_entity_certificate_not_valid_after,
tls_end_entity_certificate_not_valid_before,
tls_certificate_chain_length,
tls_failure,
http_request_url,
http_failure,
http_runtime,
failure.1 as failure_class,
IF(failure_class = 'ok', 'ok', concat(failure_class, '.', failure_str)) as failure_str_full,
IF(startsWith(failure.2, 'unknown_failure'), 'unknown_failure', failure.2) as failure_str,
failure.2 as failure_str_raw
FROM obs_web
WHERE measurement_start_time > %(measurement_start_day)s
AND measurement_start_time < %(measurement_end_day)s
AND probe_cc IN %(cc_list)s
AND hostname IN %(domain)s
""", **{
    "measurement_start_day": START_DAY,
    "measurement_end_day": END_DAY,
    "cc_list": ANALYSIS_COUNTRY_CODES,
    "domain": ANALYSIS_DOMAINS
})

Starting query_id: oonidata-9663a3e2-0348-44b4-a049-c556f2fc09a4


In [ ]:
# remove iv6 measurements
df = df[~df['ip'].str.contains('::', na=False)]
df = df[~df['ip'].str.contains(':', na=False)]

In [ ]:
df['analysis'] = df.progress_apply(compute_analysis, axis=1)

In [ ]:
df_asn_count = df[['measurement_uid', 'probe_asn']].groupby('probe_asn', as_index=False).count().rename(columns={'measurement_uid': 'obs_count'})

In [ ]:
# gives us a measure of the top ASNs for the country in terms of observation count
df_asn_count = df_asn_count.sort_values(by='obs_count', ascending=False)
df_asn_count

In [ ]:
# We select the top ASNs where we see measurements for the domains
ASN = [
  58224,
  44244,
  43754,
  197207,
  50810,
  31549,
  202468,
  206065,
  48715,
  56402
]

In [ ]:
# filter dataframe based on ASNs we want to plot for
df_asn = df[df['probe_asn'].isin(ASN)]

In [ ]:
df_agg = df_asn[[
    'measurement_start_time',
    'probe_as_org_name',
    'hostname',
    'analysis',
    'probe_asn',
    'ip',
    'resolver_asn',
    'resolver_as_org_name',
    'network_type',
    'measurement_uid'
]].groupby([
    pd.Grouper(freq='w', key='measurement_start_time'),
    'probe_as_org_name',
    'probe_asn',
    'hostname',
    'analysis',
]).count().reset_index().rename(columns={'measurement_uid': 'obs_count'})

In [660]:
# Accessible domains
analysis_domains = [
  "cahiersdufeminisme.com",
  "iraws.ir",
  "harasswatch.com",
  "zananemrooz.com",
  "jwica.ut.ac.ir",
  "hamamoun.org",
  "iwontario.com",
  "femena.net"
]

In [662]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]
# we plot with only the major blocking methodologies, and skip the ones with low measurement volume 
# for clarity
analysis_filters = [
    'dns.bogon',
    'tls.timeout',
    'tcp.timeout',
    'ok'
]
df_filter = df_filter[df_filter['analysis'].isin(analysis_filters)]

In [ ]:
present_failures = df_filter["analysis"].unique().tolist()
colors = { 
            'ok': '#37b24d', 
            'tls.timeout': '#fcc419', 
            'tcp.timeout': '#c92a2a', 
            'dns.bogon': '#be0aff', 
            'not_connected': '#ff8300'
         } 

main_chart = alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=present_failures,
            range=[colors[k] for k in present_failures]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis", "ip", "resolver_asn"]
).properties(
    width=500, 
    height=200,
    title=alt.TitleParams(
        text="Accessible Persian domains in Iran across major ASNs, Feb'24 - Oct'25",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [666]:
# Blocked domains
analysis_domains = [
  "iranwire.com",
  "zaagaah.com",
  "www.aasoo.org",
  "anfpersian.com",
  "feministschool.com",
  "www.we-change.org",
  "avishanx.com",
]

In [668]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]
# we plot with only the major blocking methodologies, and skip the ones with low measurement volume 
# for clarity
analysis_filters = [
    'dns.bogon',
    'tls.timeout',
    'tcp.timeout',
    'tls.connection_reset',
    'ok'
]
df_filter = df_filter[df_filter['analysis'].isin(analysis_filters)]

In [ ]:
present_failures = df_filter["analysis"].unique().tolist()
colors = { 
            'ok': '#37b24d', 
            'tls.timeout': '#fcc419', 
            'dns.bogon': '#e03131', 
            'tcp.timeout': '#be0aff', 
            'tls.connection_reset': '#ff8300'
         } 

main_chart = alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=present_failures,
            range=[colors[k] for k in present_failures]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis", "ip", "resolver_asn"]
).properties(
    width=500, 
    height=200,
    title=alt.TitleParams(
        text="Blocked Persian domains in Iran across major ASNs, Feb'24 - Oct'25",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [635]:
analysis_domains = [
    "https://ikwro.org.uk/",
    "https://thepolygon.ca/",
    "https://www.awid.org/",
    "https://www.iwraw-ap.org/",
    "https://learningpartnership.org/",
    "https://www.womenforwomen.org/",
    "https://www.globalfundforwomen.org/",
    "https://www.genderit.org/",
    "https://www.madre.org/",
    "https://iknowpolitics.org/",
    "https://www.wluml.org/",
    "https://feministfrequency.com/",
    "https://stopstreetharassment.org/"
]

In [ ]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]

alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color("analysis"),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis"]
).properties(
    width=400, 
    height=200,
    title = ""
).resolve_scale(
    y='independent'
)

### International Domains

In [670]:
ANALYSIS_COUNTRY_CODES = [
    'IR'
]
START_DAY = '2024-02-01'
END_DAY = '2025-10-01'

In [ ]:
# Internation domains
ANALYSIS_DOMAINS = [
    "ikwro.org.uk",
    "thepolygon.ca",
    "www.awid.org",
    "www.iwraw-ap.org",
    "learningpartnership.org",
    "www.womenforwomen.org",
    "www.globalfundforwomen.org",
    "www.genderit.org",
    "www.madre.org",
    "iknowpolitics.org",
    "www.wluml.org",
    "feministfrequency.com",
    "stopstreetharassment.org",
    "www.girlsnotbrides.org",
    "www.womenonweb.org",
    "www.nwci.ie",
    "womenhelp.org",
    "www.endfgm.eu",
    "womensmediacenter.com",
    "womenpeacemakersprogram.org",
    "www.isiswomen.org",
    "musalaha.org",
    "genderandaids.unwomen.org",
    "womenforafghanwomen.org",
    "www.rawa.org"
]

In [676]:
%%time
df = click_query("""
WITH multiIf(
    dns_failure IS NOT NULL, tuple('dns', dns_failure),
    tcp_failure IS NOT NULL, tuple('tcp', tcp_failure),
    tls_failure IS NOT NULL, tuple('tls', tls_failure),
    http_failure IS NOT NULL, tuple('https', http_failure),
    tuple('ok', '')
) as failure
SELECT 
report_id,
input,
measurement_uid,
probe_cc,
probe_asn,
probe_as_org_name,
probe_as_cc,
network_type,
measurement_start_time,
hostname,
ip,
port,
ip_asn,
ip_as_org_name,
resolver_ip,
resolver_cc,
resolver_asn,
resolver_as_org_name,
resolver_as_cc,
dns_engine,
dns_failure,
dns_answer,
tcp_success,
tcp_failure,
tls_handshake_time,
tls_handshake_read_count,
tls_handshake_write_count,
tls_handshake_read_bytes,
tls_handshake_write_bytes,
tls_handshake_last_operation,
tls_cipher_suite IS NOT NULL as tls_success,
tls_is_certificate_valid,
tls_end_entity_certificate_subject,
tls_end_entity_certificate_subject_common_name,
tls_end_entity_certificate_issuer,
tls_end_entity_certificate_issuer_common_name,
tls_end_entity_certificate_san_list,
tls_end_entity_certificate_not_valid_after,
tls_end_entity_certificate_not_valid_before,
tls_certificate_chain_length,
tls_failure,
http_request_url,
http_failure,
http_runtime,
failure.1 as failure_class,
IF(failure_class = 'ok', 'ok', concat(failure_class, '.', failure_str)) as failure_str_full,
IF(startsWith(failure.2, 'unknown_failure'), 'unknown_failure', failure.2) as failure_str,
failure.2 as failure_str_raw
FROM obs_web
WHERE measurement_start_time > %(measurement_start_day)s
AND measurement_start_time < %(measurement_end_day)s
AND probe_cc IN %(cc_list)s
AND hostname IN %(domain)s
""", **{
    "measurement_start_day": START_DAY,
    "measurement_end_day": END_DAY,
    "cc_list": ANALYSIS_COUNTRY_CODES,
    "domain": ANALYSIS_DOMAINS
})

Starting query_id: oonidata-41ed3f34-7420-4966-9ae1-2d248594a06b
CPU times: user 58.1 s, sys: 461 ms, total: 58.5 s
Wall time: 1min 53s


In [677]:
# remove ipv6 measurements
df = df[~df['ip'].str.contains('::', na=False)]
df = df[~df['ip'].str.contains(':', na=False)]

In [678]:
df['analysis'] = df.progress_apply(compute_analysis, axis=1)

100%|██████████| 136160/136160 [00:02<00:00, 53123.53it/s] 


In [679]:
df_asn_count = df[['measurement_uid', 'probe_asn']].groupby('probe_asn', as_index=False).count().rename(columns={'measurement_uid': 'obs_count'})

In [ ]:
# gives us a measure of the top ASNs for the country in terms of observation count
df_asn_count = df_asn_count.sort_values(by='obs_count', ascending=False)
df_asn_count

In [686]:
# We select the top ASNs where we see measurements for the domains
ASN = [
    58224,
    44244,
    43754,
    197207,
    50810,
    31549,
    57218,
    56402,
    39501
]

In [687]:
df_asn = df[df['probe_asn'].isin(ASN)]

In [688]:
df_agg = df_asn[[
    'measurement_start_time',
    'probe_as_org_name',
    'hostname',
    'analysis',
    'probe_asn',
    'ip',
    'resolver_asn',
    'resolver_as_org_name',
    'network_type',
    'measurement_uid'
]].groupby([
    pd.Grouper(freq='w', key='measurement_start_time'),
    'probe_as_org_name',
    'probe_asn',
    'hostname',
    'ip',
    'resolver_asn',
    'resolver_as_org_name',
    'analysis',
    'network_type'
]).count().reset_index().rename(columns={'measurement_uid': 'obs_count'})

/tmp/ipykernel_1406510/3895296698.py:13: FutureWarning: 'w' is deprecated and will be removed in a future version, please use 'W' instead.
  pd.Grouper(freq='w', key='measurement_start_time'),


In [689]:
# Blocked domains
analysis_domains = [
    "thepolygon.ca",
    "www.awid.org",
    "www.wluml.org",
    "www.womenonweb.org",
    "www.rawa.org"
]

In [ ]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]
# we plot with only the major blocking methodologies, and skip the ones with low measurement volume 
# for clarity
analysis_filters = [
    'dns.bogon',
    'tls.timeout',
    'tcp.timeout',
    'tls.connection_reset',
    'ok'
]
df_filter = df_filter[df_filter['analysis'].isin(analysis_filters)]

In [ ]:
present_failures = df_filter["analysis"].unique().tolist()
colors = { 
            'ok': '#37b24d', 
            'tls.timeout': '#fcc419', 
            'dns.bogon': '#e03131', 
            'tls.connection_reset': '#be0aff', 
            'tcp.timeout': '#ff8300'
         } 

main_chart = alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=present_failures,
            range=[colors[k] for k in present_failures]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis", "ip", "resolver_asn"]
).properties(
    width=500, 
    height=200,
    title=alt.TitleParams(
        text="Blocked International domains in Iran across major ASNs, Feb'24 - Oct'25",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [696]:
# Accessible domains
analysis_domains = [
  "www.nwci.ie",
  "womenhelp.org",
  "www.endfgm.eu",
  "genderandaids.unwomen.org",
  "womenforafghanwomen.org",
]

In [697]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]
# we plot with only the major blocking methodologies, and skip the ones with low measurement volume 
# for clarity
analysis_filters = [
    'tls.timeout',
    'tcp.timeout',
    'tls.eof_error',
    'ok'
]
df_filter = df_filter[df_filter['analysis'].isin(analysis_filters)]

In [ ]:
present_failures = df_filter["analysis"].unique().tolist()
colors = { 
            'ok': '#37b24d', 
            'tls.timeout': '#fcc419', 
            'tls.eof_error': '#e03131', 
            'tls.connection_reset': '#be0aff', 
            'tcp.timeout': '#ff8300'
         } 

main_chart = alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=present_failures,
            range=[colors[k] for k in present_failures]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis", "ip", "resolver_asn"]
).properties(
    width=500, 
    height=200,
    title=alt.TitleParams(
        text="Accessible International domains in Iran across major ASNs, Feb'24 - Oct'25",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [700]:
# Accessible domains
analysis_domains = [
  "www.iwraw-ap.org",
  "www.globalfundforwomen.org",
  "www.genderit.org",
  "www.madre.org",
  "www.girlsnotbrides.org"
]

In [701]:
df_filter = df_agg[df_agg['hostname'].isin(analysis_domains)]
# we plot with only the major blocking methodologies, and skip the ones with low measurement volume 
# for clarity
analysis_filters = [
    'tls.timeout',
    'tcp.timeout',
    'tls.eof_error',
    'ok'
]
df_filter = df_filter[df_filter['analysis'].isin(analysis_filters)]

In [ ]:
present_failures = df_filter["analysis"].unique().tolist()
colors = { 
            'ok': '#37b24d', 
            'tls.timeout': '#fcc419', 
            'tls.eof_error': '#e03131', 
            'tls.connection_reset': '#be0aff', 
            'tcp.timeout': '#ff8300'
         } 

main_chart = alt.Chart(df_filter).mark_bar(point=True).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn:N", title="asn"),
    column=alt.Column("hostname"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=present_failures,
            range=[colors[k] for k in present_failures]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "probe_asn", "hostname", "obs_count", "analysis", "ip", "resolver_asn"]
).properties(
    width=500, 
    height=200,
    title=alt.TitleParams(
        text="Accessible International domains in Iran across major ASNs, Feb'24 - Oct'25",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)